# Fabric Client — Iteration Examples

Walk through different patterns for fetching workspace metadata:
serial ``async for`` vs. concurrent ``asyncio.gather`` + ``TaskGroup``.

## Setup & Authentication

Load credentials from environment variables and build the client via
the dependency-injection container.  Log level is set to ``INFO`` to
show operational highlights (scan progress, scoped fetches, timings).

In [ ]:
from collections import defaultdict

import pandas as pd
from dotenv import load_dotenv

from fabric_client import Credentials, FabricContainer, scan

# Load credentials from .env (FABRIC_CLI_TENANT_ID, FABRIC_CLI_CLIENT_ID,
# FABRIC_CLI_CLIENT_SECRET)
load_dotenv()

creds = Credentials.env()

container = FabricContainer()
container.credentials.override(creds)
client = container.client()
client.set_log_level("INFO")

def display(d: defaultdict, name: str) -> pd.DataFrame:
    """Show a collected data category as a DataFrame."""
    return pd.DataFrame(d[name])

## Serial Iteration (``async for``)

Each nested ``async for`` blocks until the inner collection is fully
fetched.  Simple and readable, but every level runs sequentially.

In [ ]:
# Fetch all metadata in nested serial async-for loops

data = defaultdict(list)
filtered = [w for w in await client.workspaces if w.display_name is not None]
async for workspace in scan(filtered):
    data['workspaces'].append(workspace.model_dump())

    async for dataset in workspace.datasets:
        data['datasets'].append(dataset.model_dump())
        async for query in dataset.queries:
            data['queries'].append(query.model_dump())
        async for refresh in dataset.refreshes:
            data['refreshes'].append(refresh.model_dump())

    async for dataflow in workspace.dataflows:
        data["dataflows"].append(dataflow.model_dump())
        async for query in dataflow.queries:
            data["queries"].append(query)
        async for transaction in dataflow.transactions:
            data['transactions'].append(transaction.model_dump())

    async for report in workspace.reports:
        data['reports'].append(report.model_dump())

## Concurrent Iteration (``gather`` + ``TaskGroup``)

Fetch datasets, dataflows, and reports **in parallel** per workspace,
then fan out child resources (refreshes, queries, transactions) inside
an ``asyncio.TaskGroup``.  Total time ≈ slowest request, not sum of all.

In [ ]:
# Fetch datasets/dataflows/reports in parallel, then fan out child resources
import asyncio
from typing import Any

concurrent_data = defaultdict(list)

async def _gather_dataset(ds) -> tuple[Any]:
    """Fetch refreshes and queries for a single dataset concurrently."""
    refreshes, queries = await asyncio.gather(ds.refreshes, ds.queries)
    return ds, refreshes, queries

async def _gather_dataflow(df) -> tuple[Any]:
    """Fetch queries and transactions for a single dataflow concurrently
    (Gen2 dataflows return empty lists).
    """  # noqa: D205
    queries, transactions = await asyncio.gather(df.queries, df.transactions)
    return df, queries, transactions

_filtered = [w for w in await client.workspaces if w.display_name is not None]
async for workspace in scan(_filtered):
    concurrent_data["workspaces"].append(workspace.model_dump())

    # Step 1: fetch datasets, dataflows, reports in parallel
    datasets, dataflows, reports = await asyncio.gather(
        workspace.datasets, workspace.dataflows, workspace.reports,
    )

    # Step 2: fan out refreshes + queries for every dataset
    async with asyncio.TaskGroup() as tg:
        ds_tasks = [tg.create_task(_gather_dataset(ds)) for ds in datasets]
    for task in ds_tasks:
        ds, refreshes, queries = task.result()
        concurrent_data["datasets"].append(ds.model_dump())
        for r in refreshes:
            concurrent_data["refreshes"].append(r.model_dump())
        for q in queries:
            concurrent_data["queries"].append(q.model_dump())

    # Step 3: fan out queries + transactions for every dataflow
    async with asyncio.TaskGroup() as tg:
        df_tasks = [tg.create_task(_gather_dataflow(df)) for df in dataflows]
    for task in df_tasks:
        df, queries, transactions = task.result()
        concurrent_data["dataflows"].append(df.model_dump())
        for q in queries:
            concurrent_data["queries"].append(q.model_dump())
        for t in transactions:
            concurrent_data["transactions"].append(t.model_dump())

    # Step 4: reports have no sub-resources — collect directly
    for r in reports:
        concurrent_data["reports"].append(r.model_dump())

print(
    f"datasets={len(concurrent_data['datasets'])}, "
    f"dataflows={len(concurrent_data['dataflows'])}, "
    f"reports={len(concurrent_data['reports'])}, "
    f"refreshes={len(concurrent_data['refreshes'])}, "
    f"transactions={len(concurrent_data['transactions'])}, "
    f"queries={len(concurrent_data['queries'])}"
)

## View Results

Use the ``display()`` helper to inspect collected data as DataFrames.
Change ``concurrent_data`` to ``data`` to view the serial results.

In [ ]:
display(concurrent_data,"workspaces")